<a href="https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Two signals behind my rule, both tied to real FlyRank flags:

Signal A — staleness (behind stale_visible_page): freshness_tier vs. decline rate, with a volume floor (impressions_90d ≥ 100, since thin pages make any rate noisy). Decline rate rises monotonically with staleness — 58.3% (0-30d, n=13,735) → 59.2% (31-90d, n=152) → 62.2% (91-180d, n=8,084) → 74.3% (181+d, n=35). Verdict: CONFIRMED — direction holds cleanly across all four tiers, though the two middle/tail buckets are thin (n=152, n=35) and shouldn't be read as precisely as the two large ones.

Signal B — CTR vs. position (behind CTR-fix logic): median CTR by position_tier, same volume floor. page_1=0.23% (n=8,633) → striking=0.15% (n=5,903) → page_3_5=0.06% (n=6,058) → deep=0.00% (n=879) — a clean monotonic collapse. The one wrinkle: top_3=0.19% (n=533) is lower than page_1. That's a real, small-n bucket, not obviously noise — top-3 results often sit under knowledge panels/PAA boxes that eat clicks even at position 1-3. Verdict: MIXED — position clearly drives CTR overall, but the very top of the funnel inverts the expected order.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

url = "https://raw.githubusercontent.com/Purvansh09/flyrannk_week1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

vol_floor = df['impressions_90d'] >= 100

# Signal A — staleness vs decline rate
sigA = (df[vol_floor].groupby('freshness_tier')
        .agg(n=('content_id','count'), decline_rate=('is_declining_label','mean'))
        .reindex(['0-30','31-90','91-180','181+']).round(3))
print("Signal A — freshness_tier vs decline_rate (impressions_90d>=100)")
print(sigA)

# Signal B — CTR by position tier
order = ['top_3','page_1','striking','page_3_5','deep']
sigB = (df[vol_floor & df['position_tier'].isin(order)].groupby('position_tier')
        .agg(n=('content_id','count'), median_ctr=('ctr','median'), mean_ctr=('ctr','mean'))
        .reindex(order).round(3))
print("\nSignal B — position_tier vs ctr (impressions_90d>=100)")
print(sigB)


Signal A — freshness_tier vs decline_rate (impressions_90d>=100)
                    n  decline_rate
freshness_tier                     
0-30            13735         0.583
31-90             152         0.592
91-180           8084         0.622
181+               35         0.743

Signal B — position_tier vs ctr (impressions_90d>=100)
                  n  median_ctr  mean_ctr
position_tier                            
top_3           533        0.19     0.334
page_1         8633        0.23     0.355
striking       5903        0.15     0.256
page_3_5       6058        0.06     0.142
deep            879        0.00     0.055


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Plain words first: A page is worth reviewing if it hasn't been touched in a while AND its click-through rate is falling well short of what pages in its search position normally get.

Score (no fitted weights — both terms are hand-chosen, 50/50): 50 × staleness_score + 50 × ctr_gap_score, computed only where impressions_90d ≥ 100 and position data exists (the volume floor Signal A/B both needed). Rows ≥30 get reason code stale_ctr_underperformer and action refresh_review.

Evaluated at K, against is_declining_label (the closest available proxy for "was this the right page to flag"): base rate on this slice is 59.8%. The rule gets precision@20 = 0.95 (19/20), precision@50 = 0.88 (44/50) — well above the coin-flip-ish base rate, so it's not just picking up the majority class for free. I'm not claiming this beats a trained model (the repo's own numbers show random forest beating the product's baseline); I'm just showing this hand-rule is meaningfully better than guessing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

order = ['top_3','page_1','striking','page_3_5','deep']
benchmark = sigB['median_ctr'].to_dict()
df['expected_ctr'] = df['position_tier'].map(benchmark)

df['ctr_gap_score'] = ((df['expected_ctr'] - df['ctr']) / df['expected_ctr']).clip(lower=0, upper=1)
df['staleness_score'] = (df['days_since_last_update'] / 365).clip(upper=1)

eligible = df['position_tier'].isin(order) & vol_floor
df['baseline_action_score'] = 0.0
df.loc[eligible, 'baseline_action_score'] = (
    50 * df.loc[eligible, 'staleness_score'] + 50 * df.loc[eligible, 'ctr_gap_score']
).round(1)

flag_thresh = 30
df['reason_code'] = ''
df['action'] = 'monitor'
flagged = eligible & (df['baseline_action_score'] >= flag_thresh)
df.loc[flagged, 'reason_code'] = 'stale_ctr_underperformer'
df.loc[flagged, 'action'] = 'refresh_review'
df.loc[~eligible, 'action'] = 'insufficient_data'

print("eligible:", eligible.sum(), "/", len(df), "| flagged:", flagged.sum())

ranked = df[eligible].sort_values('baseline_action_score', ascending=False).reset_index(drop=True)
cols_out = ['content_id','client_id','baseline_action_score','reason_code','action',
            'position_tier','avg_position','ctr','expected_ctr','ctr_gap_score',
            'days_since_last_update','freshness_tier','staleness_score',
            'impressions_90d','trend_direction']

os.makedirs('work/outputs', exist_ok=True)
ranked[cols_out].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("wrote work/outputs/baseline_action_score.csv —", len(ranked), "rows")

# precision@K, per the building-baselines skill: always print base rate alongside it
def precision_at_k(ranked_df, label_col, k):
    return ranked_df[label_col].head(k).mean()

base_rate = ranked['is_declining_label'].mean()
print(f"\nbase rate (is_declining_label): {base_rate:.3f}")
for k in (20, 50, 100):
    p = precision_at_k(ranked, 'is_declining_label', k)
    print(f"precision@{k}: {p:.3f}  ({int(round(p*k))}/{k})")

# dummy floor-below-the-floor, per skill's "Also useful"
majority_rate = max(base_rate, 1 - base_rate)
print(f"\ndummy majority-class baseline: {majority_rate:.3f}  <-- your rule needs to clear this")

eligible: 22006 / 30000 | flagged: 7639
wrote work/outputs/baseline_action_score.csv — 22006 rows

base rate (is_declining_label): 0.598
precision@20: 0.950  (19/20)
precision@50: 0.840  (42/50)
precision@100: 0.770  (77/100)

dummy majority-class baseline: 0.598  <-- your rule needs to clear this


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20, printed below: the action, why (staleness + CTR gap numbers), and what would make it wrong.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20).copy()

def make_why(r):
    return (f"stale ({int(r['days_since_last_update'])}d since update, tier {r['freshness_tier']}) "
            f"+ CTR {r['ctr']:.2f}% vs {r['expected_ctr']:.2f}% expected at position {r['avg_position']:.1f} "
            f"({r['position_tier']}) -> score {r['baseline_action_score']:.1f}")

def make_wrong(r):
    if r['trend_direction'] != 'down':
        return f"trend_direction is '{r['trend_direction']}', not declining — low CTR may reflect query type, not a fixable problem"
    if r['days_since_last_update'] in (106, 183):
        return "shares its exact update-age with several other rows from the same client — looks like a bulk timestamp, not individual neglect"
    return "if the low CTR reflects informational intent (no need to click through), a refresh may not move CTR"

top20['why'] = top20.apply(make_why, axis=1)
top20['what_would_make_it_wrong'] = top20.apply(make_wrong, axis=1)

for i, r in top20.iterrows():
    print(f"{i+1}. [{r['action']}] {r['content_id']} — {r['why']} | Wrong if: {r['what_would_make_it_wrong']}")

1. [refresh_review] content_02b0d6e30129 — stale (313d since update, tier 181+) + CTR 0.00% vs 0.23% expected at position 6.9 (page_1) -> score 92.9 | Wrong if: if the low CTR reflects informational intent (no need to click through), a refresh may not move CTR
2. [refresh_review] content_f488400fca67 — stale (305d since update, tier 181+) + CTR 0.00% vs 0.23% expected at position 5.7 (page_1) -> score 91.8 | Wrong if: if the low CTR reflects informational intent (no need to click through), a refresh may not move CTR
3. [refresh_review] content_ab27c30d81f4 — stale (304d since update, tier 181+) + CTR 0.00% vs 0.23% expected at position 8.9 (page_1) -> score 91.6 | Wrong if: trend_direction is 'stable', not declining — low CTR may reflect query type, not a fixable problem
4. [refresh_review] content_4f241bad48a3 — stale (236d since update, tier 181+) + CTR 0.00% vs 0.15% expected at position 19.1 (striking) -> score 82.3 | Wrong if: if the low CTR reflects informational intent (no need 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks. Two patterns stand out in the top 20: (1) 14 of the 20 slots come from just two clients (client_7f2253d7e2 ×9, client_d029fa3a95 ×5), and within each client every pick shares the exact same days_since_last_update (106 or 183 days) — that's a bulk platform timestamp, not evidence any one of those pages was individually neglected. (2) Rank 3 (content_ab27c30d81f4) has trend_direction == 'stable', not down — the rule surfaced it purely on staleness + CTR gap, since trend was deliberately excluded from the score. Both are legitimate "what would make it wrong" cases, not bugs — but they mean a human reviewer should sanity-check client concentration before batch-actioning the top of the queue.

Leakage check. The score uses only days_since_last_update, ctr, avg_position/position_tier, and impressions_90d as a volume floor — all observed, pre-decision signals from the same 90-day window, none computed after any decision point. trend_direction/trend_pct (the label source) never enter the formula — they were used only in Section 1's signal audit, never as a feature. No product flags (health_score, priority_score, action_type) exist in this dataset to begin with, so there's nothing to accidentally copy. No future window exists in this 90-day snapshot to overlap with.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Client concentration in top 20:")
print(top20['client_id'].value_counts())

print("\ntrend_direction in top 20:")
print(top20['trend_direction'].value_counts())

print("\nConfirming no leakage columns in the score inputs:")
score_inputs = {'days_since_last_update','ctr','avg_position','position_tier','impressions_90d'}
leak_cols = {'trend_direction','trend_pct'}
print("overlap (should be empty):", score_inputs & leak_cols)

Client concentration in top 20:
client_id
client_7f2253d7e2    7
client_d029fa3a95    5
client_f369cb89fc    3
client_19581e27de    1
client_6208ef0f77    1
client_4ec9599fc2    1
client_9400f1b21c    1
client_9f14025af0    1
Name: count, dtype: int64

trend_direction in top 20:
trend_direction
down      19
stable     1
Name: count, dtype: int64

Confirming no leakage columns in the score inputs:
overlap (should be empty): set()


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.